In [2]:
# Reload merged data with notes
import pandas as pd
df_notes = pd.read_csv('ratings_clean_v3.csv')

# Check notes field
total = len(df_notes)
with_notes = df_notes['notes'].notna() & (df_notes['notes'].str.strip() != '')
print(f"Total ratings: {total}")
print(f"Ratings with notes: {with_notes.sum()}")
print(f"Percentage with notes: {with_notes.sum()/total*100:.1f}%")
print(f"\nRatings without notes: {(~with_notes).sum()}")

# Notes by rater
print("\nNotes per rater:")
notes_by_rater = df_notes[with_notes].groupby('evaluator_name')['notes'].count()
print(notes_by_rater.sort_values(ascending=False))

# Show all notes
print("\nAll notes:")
all_notes = df_notes[with_notes][['evaluator_name', 'prompt_idx', 'notes']]
for _, row in all_notes.iterrows():
    print(f"\nRater: {row['evaluator_name']} | Prompt: {row['prompt_idx']}")
    print(f"Note: {row['notes']}")

Total ratings: 780
Ratings with notes: 12
Percentage with notes: 1.5%

Ratings without notes: 768

Notes per rater:
evaluator_name
Rater_14    4
Rater_15    3
Rater_9     3
Rater_13    1
Rater_2     1
Name: notes, dtype: int64

All notes:

Rater: Rater_2 | Prompt: 0
Note: Model C gives a longer, more elaborate completion with two sentences. It may be slightly too assumptive about what the person wants and adds detail the user never specified.

Rater: Rater_9 | Prompt: 0
Note: Model B didn't complete the sentence, it just gave more options instead. Model C gave a normal completion but added more context to it.

Rater: Rater_9 | Prompt: 2
Note: Model B provides series of options rather than sticking to one request

Rater: Rater_9 | Prompt: 28
Note: The rest of the models seem to understand the urgency but the model B isn't giving answer based on context dependancy, an urgent situation doesn't require other options.

Rater: Rater_13 | Prompt: 39
Note: Model 2 (the model that provides a va

In [4]:
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation

wb = Workbook()
# SHEET 1: INSTRUCTIONS
ws_instructions = wb.active
ws_instructions.title = "Instructions"

instructions = [
    ("CIF QUALITATIVE CODING SHEET", ""),
    ("Study: Evaluating LLMs as Assistive Communication Tools", ""),
    ("", ""),
    ("PURPOSE", ""),
    ("This sheet contains 12 optional notes left by evaluation participants during the study.", ""),
    ("Your task is to read each note EXACTLY as written and assign CIF dimensions to it.", ""),
    ("Do NOT paraphrase or interpret — code based on what the participant actually wrote.", ""),
    ("", ""),
    ("YOUR CODING TASK", ""),
    ("For each note, assign:", ""),
    ("1. Primary CIF Dimension — the main dimension the note is commenting on", ""),
    ("2. Secondary CIF Dimension — a second dimension if relevant (can be left blank)", ""),
    ("3. Justification — briefly explain why you chose this dimension (1-2 sentences)", ""),
    ("4. AT Alignment — does the participant's understanding align with assistive tech principles?", ""),
    ("", ""),
    ("CIF DIMENSIONS — Choose from these five only:", ""),
    ("Clarity", "The ease with which the output can be understood by the intended user"),
    ("Appropriateness", "The suitability of the response in terms of language level, tone, and context for the specific disability category"),
    ("Accuracy", "The factual correctness and relevance of the output relative to the prompt"),
    ("Consistency", "The internal coherence and logical structure of the response"),
    ("Empathy", "The degree to which the response reflects respect, dignity, and person-centered language toward the user"),
    ("", ""),
    ("AT ALIGNMENT OPTIONS — Choose from these three only:", ""),
    ("Aligns", "The participant's comment reflects established assistive technology and AAC principles"),
    ("Misaligns", "The participant's comment contradicts or misunderstands assistive technology principles"),
    ("Partially Aligns", "The participant's comment has some validity but also some tension with AT principles"),
    ("", ""),
    ("IMPORTANT RULES", ""),
    ("- Read each note in full before coding", ""),
    ("- Code independently — do not discuss with other coders until all coding is complete", ""),
    ("- Use ONLY the five CIF dimensions listed above", ""),
    ("- Leave Secondary CIF blank if only one dimension applies", ""),
    ("- If unsure between two dimensions pick the one most clearly supported by the note text", ""),
    ("- Go to the Coding Sheet tab to begin", ""),
]

for row_idx, (col_a, col_b) in enumerate(instructions, 1):
    ws_instructions.cell(row=row_idx, column=1, value=col_a)
    ws_instructions.cell(row=row_idx, column=2, value=col_b)
    if col_a == "CIF QUALITATIVE CODING SHEET":
        ws_instructions.cell(row=row_idx, column=1).font = Font(bold=True, size=14, color="1F3864")
    elif col_a in ["PURPOSE", "YOUR CODING TASK", "CIF DIMENSIONS — Choose from these five only:",
                    "AT ALIGNMENT OPTIONS — Choose from these three only:", "IMPORTANT RULES"]:
        ws_instructions.cell(row=row_idx, column=1).font = Font(bold=True, size=11, color="2E86AB")
    elif col_a in ["Clarity", "Appropriateness", "Accuracy", "Consistency", "Empathy",
                    "Aligns", "Misaligns", "Partially Aligns"]:
        ws_instructions.cell(row=row_idx, column=1).font = Font(bold=True, size=10)
        ws_instructions.cell(row=row_idx, column=2).font = Font(italic=True, size=10)

ws_instructions.column_dimensions['A'].width = 45
ws_instructions.column_dimensions['B'].width = 80

# SHEET 2: CODING SHEET
ws_coding = wb.create_sheet("Coding Sheet")

notes = [
    {
        'Note_ID': 'N01',
        'Rater_ID': 'Rater_2',
        'Prompt_Number': 'Prompt 0',
        'Model_Referenced': 'Model C (Groq/Llama 3)',
        'Note_Text_Exact_As_Written': 'Model C gives a longer, more elaborate completion with two sentences. It may be slightly too assumptive about what the person wants and adds detail the user never specified.',
    },
    {
        'Note_ID': 'N02',
        'Rater_ID': 'Rater_9',
        'Prompt_Number': 'Prompt 0',
        'Model_Referenced': 'Model B (Claude) and Model C (Groq/Llama 3)',
        'Note_Text_Exact_As_Written': "Model B didn't complete the sentence, it just gave more options instead. Model C gave a normal completion but added more context to it.",
    },
    {
        'Note_ID': 'N03',
        'Rater_ID': 'Rater_9',
        'Prompt_Number': 'Prompt 2',
        'Model_Referenced': 'Model B (Claude)',
        'Note_Text_Exact_As_Written': 'Model B provides series of options rather than sticking to one request',
    },
    {
        'Note_ID': 'N04',
        'Rater_ID': 'Rater_9',
        'Prompt_Number': 'Prompt 28',
        'Model_Referenced': 'Model B (Claude)',
        'Note_Text_Exact_As_Written': "The rest of the models seem to understand the urgency but the model B isn't giving answer based on context dependancy, an urgent situation doesn't require other options.",
    },
    {
        'Note_ID': 'N05',
        'Rater_ID': 'Rater_13',
        'Prompt_Number': 'Prompt 39',
        'Model_Referenced': 'Model B (Claude)',
        'Note_Text_Exact_As_Written': 'Model 2 (the model that provides a variety of answers) tends to allow greater accuracy across the five dimensions.',
    },
    {
        'Note_ID': 'N06',
        'Rater_ID': 'Rater_14',
        'Prompt_Number': 'Prompt 3',
        'Model_Referenced': 'Model C (Groq/Llama 3)',
        'Note_Text_Exact_As_Written': 'The two parts of model C are inconsistent.',
    },
    {
        'Note_ID': 'N07',
        'Rater_ID': 'Rater_14',
        'Prompt_Number': 'Prompt 7',
        'Model_Referenced': 'Model C (Groq/Llama 3)',
        'Note_Text_Exact_As_Written': 'I think the second sentence in Model C is not necessary',
    },
    {
        'Note_ID': 'N08',
        'Rater_ID': 'Rater_14',
        'Prompt_Number': 'Prompt 25',
        'Model_Referenced': 'General',
        'Note_Text_Exact_As_Written': 'More casual will work just fine',
    },
    {
        'Note_ID': 'N09',
        'Rater_ID': 'Rater_14',
        'Prompt_Number': 'Prompt 29',
        'Model_Referenced': 'Model C (Groq/Llama 3)',
        'Note_Text_Exact_As_Written': "He shouldn't be missing his wife if he's home. It should be along the lines of the person missing his wife and wanting to go home instead.",
    },
    {
        'Note_ID': 'N10',
        'Rater_ID': 'Rater_15',
        'Prompt_Number': 'Prompt 1',
        'Model_Referenced': 'Model B (Claude)',
        'Note_Text_Exact_As_Written': "If accuracy means the output should be factually correct and true to the prompt, then rating model B is quite dicey. This is because the prompt says 1 - 2 sentences but B always provides about 4 options and 5 - 6 sentences.\n\nModel B is probably going to be the most accurate for the user because it gives options unlike the others, but that's against the prompt given and rating rubic.",
    },
    {
        'Note_ID': 'N11',
        'Rater_ID': 'Rater_15',
        'Prompt_Number': 'Prompt 4',
        'Model_Referenced': 'Model B (Claude), Model A (GPT-4), Model C (Groq/Llama 3)',
        'Note_Text_Exact_As_Written': "For my understanding of what was written, empathy means respecting the users dignity and autonomy which means their ability to make choices or initiate conversation without someone taking over from the. Option B does that because now they have a range of sentences to select from unlike the rest that doesn't really give them a choice.\n\nAnd in my opinion empathy from model A and C is not that great when it comes to AAC",
    },
    {
        'Note_ID': 'N12',
        'Rater_ID': 'Rater_15',
        'Prompt_Number': 'Prompt 6',
        'Model_Referenced': 'Model B (Claude) and Model C (Groq/Llama 3)',
        'Note_Text_Exact_As_Written': "I fear that my idea of accuracy is wrong because model C doesn't seem accurate if it's based on trying to figure out the exact needs of the user.\n\nSince the prompt however said 1 - 2 sentence and to keep it natural and clear, then model C is accurate in that regard and model B isn't.",
    },
]

headers = [
    'Note_ID', 'Rater_ID', 'Prompt_Number', 'Model_Referenced',
    'Note_Text_Exact_As_Written',
    'Coder_1_Primary_CIF', 'Coder_1_Secondary_CIF', 'Coder_1_AT_Alignment', 'Coder_1_Justification',
    'Coder_2_Primary_CIF', 'Coder_2_Secondary_CIF', 'Coder_2_AT_Alignment', 'Coder_2_Justification',
    'Coder_3_Primary_CIF', 'Coder_3_Secondary_CIF', 'Coder_3_AT_Alignment', 'Coder_3_Justification',
    'AGREED_Primary_CIF', 'AGREED_Secondary_CIF', 'AGREED_AT_Alignment', 'Consensus_Notes',
]

for col_idx, header in enumerate(headers, 1):
    cell = ws_coding.cell(row=1, column=col_idx, value=header)
    cell.fill = PatternFill(start_color="1F3864", end_color="1F3864", fill_type="solid")
    cell.font = Font(color="FFFFFF", bold=True, size=10)
    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

for row_idx, note in enumerate(notes, 2):
    values = [
        note['Note_ID'], note['Rater_ID'], note['Prompt_Number'],
        note['Model_Referenced'], note['Note_Text_Exact_As_Written'],
        '', '', '', '',
        '', '', '', '',
        '', '', '', '',
        '', '', '', '',
    ]
    for col_idx, value in enumerate(values, 1):
        cell = ws_coding.cell(row=row_idx, column=col_idx, value=value)
        cell.alignment = Alignment(vertical='top', wrap_text=True)
        header = headers[col_idx - 1]
        if 'Coder_1' in header:
            cell.fill = PatternFill(start_color="D6E4F0", end_color="D6E4F0", fill_type="solid")
        elif 'Coder_2' in header:
            cell.fill = PatternFill(start_color="FCE4EC", end_color="FCE4EC", fill_type="solid")
        elif 'Coder_3' in header:
            cell.fill = PatternFill(start_color="E8F5E9", end_color="E8F5E9", fill_type="solid")
        elif 'AGREED' in header or 'Consensus' in header:
            cell.fill = PatternFill(start_color="FFF9C4", end_color="FFF9C4", fill_type="solid")
        if col_idx == 1:
            cell.font = Font(bold=True)

# Dropdown validation
cif_options = '"Clarity,Appropriateness,Accuracy,Consistency,Empathy"'
at_options = '"Aligns,Misaligns,Partially Aligns"'

cif_cols = [6, 7, 10, 11, 14, 15, 18, 19]
at_cols = [8, 12, 16, 20]

for col in cif_cols:
    dv = DataValidation(type="list", formula1=cif_options, allow_blank=True)
    ws_coding.add_data_validation(dv)
    for row in range(2, 14):
        dv.add(ws_coding.cell(row=row, column=col))

for col in at_cols:
    dv = DataValidation(type="list", formula1=at_options, allow_blank=True)
    ws_coding.add_data_validation(dv)
    for row in range(2, 14):
        dv.add(ws_coding.cell(row=row, column=col))

col_widths = {
    1: 8, 2: 12, 3: 12, 4: 25, 5: 50,
    6: 18, 7: 18, 8: 18, 9: 30,
    10: 18, 11: 18, 12: 18, 13: 30,
    14: 18, 15: 18, 16: 18, 17: 30,
    18: 18, 19: 18, 20: 18, 21: 30,
}
for col, width in col_widths.items():
    ws_coding.column_dimensions[get_column_letter(col)].width = width

for row in range(2, 14):
    ws_coding.row_dimensions[row].height = 80

ws_coding.freeze_panes = 'F2'

# SHEET 3: CIF REFERENCE
ws_ref = wb.create_sheet("CIF Reference")

ref_headers = ['CIF Dimension', 'Definition', 'What to look for in notes', 'Poor Example', 'Good Example']
for col_idx, header in enumerate(ref_headers, 1):
    cell = ws_ref.cell(row=1, column=col_idx, value=header)
    cell.fill = PatternFill(start_color="1F3864", end_color="1F3864", fill_type="solid")
    cell.font = Font(color="FFFFFF", bold=True)
    cell.alignment = Alignment(horizontal='center', wrap_text=True)

cif_ref = [
    ('Clarity',
     'The ease with which the output can be understood by the intended user',
     'Notes mentioning confusing language, complexity, verbosity, or simplicity of the response',
     'Response is convoluted, uses complex terminology, or is ambiguous',
     'Response is immediately understandable, uses plain language appropriate for the user'),
    ('Appropriateness',
     'The suitability of the response in terms of language level, tone, and context for the specific disability category',
     'Notes mentioning tone, context sensitivity, disability-specific suitability, or communication style',
     'Response uses language too complex for AAC user or ignores aphasia context',
     'Response matches the linguistic and communicative needs of the specific disability'),
    ('Accuracy',
     'The factual correctness and relevance of the output relative to the prompt',
     'Notes mentioning factual errors, prompt non-compliance, off-topic responses, or format violations',
     'Response contradicts context, ignores prompt format (e.g. gives 4 options when 1-2 sentences requested)',
     'Response correctly addresses the prompt and maintains factual integrity'),
    ('Consistency',
     'The internal coherence and logical structure of the response',
     'Notes mentioning contradictions, inconsistency between parts, or illogical structure',
     'Response contradicts itself or has inconsistent parts (e.g. two conflicting sentences)',
     'Response maintains coherent and logical structure from start to finish'),
    ('Empathy',
     'The degree to which the response reflects respect, dignity, and person-centered language toward the user',
     'Notes mentioning user autonomy, dignity, assumptions about user needs, or emotional sensitivity',
     'Response makes assumptions about user needs, ignores emotional context, or is dismissive',
     'Response centers user dignity, preserves autonomy, and acknowledges emotional context'),
]

colors = ["E3F2FD", "F3E5F5", "E8F5E9", "FFF8E1", "FCE4EC"]
for row_idx, row_data in enumerate(cif_ref, 2):
    for col_idx, value in enumerate(row_data, 1):
        cell = ws_ref.cell(row=row_idx, column=col_idx, value=value)
        cell.fill = PatternFill(start_color=colors[row_idx-2],
                                end_color=colors[row_idx-2], fill_type="solid")
        cell.alignment = Alignment(vertical='top', wrap_text=True)
        if col_idx == 1:
            cell.font = Font(bold=True)

ref_widths = [15, 45, 40, 40, 40]
for col_idx, width in enumerate(ref_widths, 1):
    ws_ref.column_dimensions[get_column_letter(col_idx)].width = width

for row in range(2, 7):
    ws_ref.row_dimensions[row].height = 80

ws_ref.freeze_panes = 'B2'

# Save
wb.save('CIF_Qualitative_Coding_Sheet_v2.xlsx')
print("Done — CIF Qualitative Coding Sheet v2 created successfully")
print("Sheets: Instructions, Coding Sheet, CIF Reference")
print("Notes: 12 — exact as written by participants")
print("Coders: 3 color-coded sections")
print("Dropdowns: CIF dimensions and AT alignment")

Done — CIF Qualitative Coding Sheet v2 created successfully
Sheets: Instructions, Coding Sheet, CIF Reference
Notes: 12 — exact as written by participants
Coders: 3 color-coded sections
Dropdowns: CIF dimensions and AT alignment
